# Hasselt v2 — Complete Improved Pipeline

All improvements integrated:
- Dimensionless feature engineering
- Group-aware data splitting (no leakage from parametric clusters)
- LayerNorm + skip connections + dropout
- HuberLoss (forces) + asymmetric FatigueLoss
- AdamW with weight decay, no scheduler
- Fixed mixup (no frozen seed)
- Experiment logger (CSV)
- SHAP analysis

## Cell 1 — Imports

In [5]:
import numpy as np
import torch
import itertools
import pandas as pd
import matplotlib.pyplot as plt
from anastruct import SystemElements
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import openpyxl as pxl
import torch.nn.functional as F
import os
import re
import csv
from datetime import datetime
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import shap

## Cell 2 — General Parameters

In [6]:
alpha      = 0.5
dropout    = 0.15
Lambda     = 0.0
epochs     = 800
seed       = 42
patience   = 80
batch_size = 16
CHECK      = ['Friction_Slider']

## Cell 3 — Import Excel Data (unchanged from your original)

In [7]:
notebook_dir = os.getcwd()
file_path = os.path.join(notebook_dir, 'Data', 'MLPartB3.xlsx')

wb = pxl.load_workbook(file_path)
ws = wb.active

headers = [ws.cell(1, col).value for col in range(1, ws.max_column + 1)]

letter_to_name = {}
for col in range(1, ws.max_column + 1):
    letter = pxl.utils.get_column_letter(col)
    letter_to_name[letter] = headers[col - 1]

def excel_to_python(formula, letter_to_name):
    result = formula
    result = result.lstrip('=')
    result = result.replace('$', '')
    result = re.sub(r'\bPI\(\)', 'np.pi', result)

    def replace_ref(match):
        letter = match.group(1)
        if letter in letter_to_name:
            name = letter_to_name[letter]
            return f"df['{name}']"
        return match.group(0)

    result = re.sub(r'\b([A-Z]{1,3})(\d+)\b', replace_ref, result)
    excel_to_np = {
        'SIN': 'np.sin', 'COS': 'np.cos', 'TAN': 'np.tan',
        'ATAN': 'np.arctan', 'ATAN2': 'np.arctan2', 'ASIN': 'np.arcsin',
        'ACOS': 'np.arccos', 'SQRT': 'np.sqrt', 'ABS': 'np.abs',
        'EXP': 'np.exp', 'LOG': 'np.log', 'LOG10': 'np.log10',
        'POWER': 'np.power', 'MOD': 'np.mod', 'FLOOR': 'np.floor',
        'CEIL': 'np.ceil', 'ROUND': 'np.round', 'MAX': 'np.maximum',
        'MIN': 'np.minimum', 'SUM': 'np.sum',
    }
    for excel_fn, np_fn in excel_to_np.items():
        result = re.sub(rf'\b{excel_fn}\b', np_fn, result)
    result = result.replace('^', '**')
    result = result.lstrip('=')
    return result

formula_list = {}
for col in range(1, ws.max_column + 1):
    cell = ws.cell(2, col)
    value = cell.value
    name = headers[col - 1]
    if isinstance(value, str) and value.startswith('='):
        formula_list[name] = excel_to_python(value, letter_to_name)

raw_cols = [h for h in headers if h not in formula_list]

def compute_features(df, formula_list):
    df = df.copy()
    remaining = dict(formula_list)
    max_passes = len(formula_list) + 1
    passes = 0
    while remaining and passes < max_passes:
        passes += 1
        newly_computed = []
        for col_name, expr in remaining.items():
            try:
                df[col_name] = eval(expr)
                newly_computed.append(col_name)
            except Exception:
                pass
        for col in newly_computed:
            del remaining[col]
        if not newly_computed:
            print(f"\n Warning Error Newly Computed")
            break
    return df

wb_data = pxl.load_workbook(file_path, data_only=True)
ws_data = wb_data.active
data = [row for row in ws_data.iter_rows(min_row=2, values_only=True)]
df_raw = pd.DataFrame(data, columns=headers)
df_computed = compute_features(df_raw[raw_cols], formula_list)
df_training = df_computed.drop(columns=[None], errors='ignore').dropna().reset_index(drop=True)
print(f"df_training shape: {df_training.shape}")

df_training shape: (946, 18)


## Cell 4 — Dimensionless Feature Engineering (NEW)

In [8]:
# Geometry ratios
df_training['e/r']   = df_training['e (m)'] / df_training['r (m)']
df_training['L/r']   = df_training['L (m)'] / df_training['r (m)']
df_training['h/w']   = df_training['height (m)'] / df_training['width (m)']
df_training['pin/w'] = df_training['pin_dia (m)'] / df_training['width (m)']
df_training['Ar/Ac'] = df_training['Ar (m²)'] / df_training['Ac (m²)']
df_training['dy/L']  = df_training['dy (m)'] / df_training['L (m)']

# Stress ratios
df_training['sigma_rod/sigma_c'] = (df_training['sigma_rod (Pa)']
                                    / df_training['sigma_c (Pa)'].clip(lower=1e-8))

# Force ratios
df_training['F_friction/F_pin'] = (df_training['F_friction (N)']
                                   / df_training['F_pin (N)'].clip(lower=1e-8))
df_training['F_result/F_pin']   = (df_training['F_resultant (N)']
                                   / df_training['F_pin (N)'].clip(lower=1e-8))
df_training['Fx/Fy']            = (df_training['Fx_pin']
                                   / df_training['Fy_pin'].abs().clip(lower=1e-8))

# Power ratios
df_training['P_friction/P_input'] = (df_training['P_friction']
                                     / df_training['P_input'].abs().clip(lower=1e-8))
df_training['P_output/P_input']   = (df_training['P_output']
                                     / df_training['P_input'].abs().clip(lower=1e-8))

# Slenderness
df_training['lambda_dy_r'] = (df_training['lambda']
                              * df_training['dy (m)']
                              / df_training['r (m)'])

dimensionless_cols = [
    'e/r', 'L/r', 'h/w', 'pin/w', 'Ar/Ac', 'dy/L',
    'sigma_rod/sigma_c', 'F_friction/F_pin', 'F_result/F_pin', 'Fx/Fy',
    'P_friction/P_input', 'P_output/P_input', 'lambda_dy_r',
]

# Verify no NaN/Inf
for col in dimensionless_cols:
    n_bad = (~np.isfinite(df_training[col])).sum()
    if n_bad > 0:
        print(f"  ⚠ {col}: {n_bad} non-finite values — clipping")
        df_training[col] = df_training[col].replace([np.inf, -np.inf], np.nan)
        df_training[col] = df_training[col].fillna(df_training[col].median())

print(f"\n  Added {len(dimensionless_cols)} dimensionless features")
print(f"  New shape: {df_training.shape}")

KeyError: 'e (m)'

## Cell 5 — Target & Feature Definitions

In [ ]:
force_cols   = ['|RT0|max', '|P1|max', '|B0|max',
                '|RT0|min', '|T1|min', '|P1|min', '|B0|min']
fatigue_cols = ['N_To', 'N_B0', 'N_P1']

target_col = force_cols + fatigue_cols
n_force    = len(force_cols)
n_fatigue  = len(fatigue_cols)
n_output   = len(target_col)

# Identify constant columns
constant_cols = {}
def is_constant(series, cv_threshold=0.01):
    std  = float(series.std())
    mean = float(series.mean())
    if pd.isna(std) or std == 0:        return True
    if mean == 0 or pd.isna(mean):      return std < 1e-10
    return (std / abs(mean)) < cv_threshold

for col in df_training.columns:
    if col is None or col in target_col:
        continue
    series = df_training[col].dropna()
    if len(series) == 0:
        continue
    if is_constant(series):
        constant_cols[col] = float(series.iloc[0])

C  = df_training[list(constant_cols.keys())].copy()
Yf = df_training[force_cols].copy()
Ya = df_training[fatigue_cols].copy()
Y  = df_training[target_col].copy()
X  = df_training.drop(columns=target_col, errors='ignore').copy()
X  = X[[col for col in X.columns if col is not None]]

feature_cols = list(X.columns)

assert np.isfinite(Ya.values).all(), \
    f"Non-finite values in Ya — check log10 columns in Excel. " \
    f"Inf count: {np.isinf(Ya.values).sum()}"

x  = torch.tensor(X.values,  dtype=torch.float32)
yf = torch.tensor(Yf.values, dtype=torch.float32)
ya = torch.tensor(Ya.values, dtype=torch.float32)
y  = torch.tensor(Y.values,  dtype=torch.float32)
c  = torch.tensor(C.values,  dtype=torch.float32)

print(f"  Dataframe")
print(f"\n  C  — Constants             : {C.shape}")
print(f"\n  Yf — Force targets (7)     : {Yf.shape}")
print(f"\n  Ya — Fatigue targets (3)   : {Ya.shape}")
print(f"\n  X  — Features              : {X.shape}")
print(f"       includes {len(dimensionless_cols)} dimensionless features")
print(f"\n  x  tensor : {x.shape}   NaN: {torch.isnan(x).sum().item()}")
print(f"  yf tensor : {yf.shape}  NaN: {torch.isnan(yf).sum().item()}")
print(f"  ya tensor : {ya.shape}  NaN: {torch.isnan(ya).sum().item()}")
print(f"\n  ya range  : min={ya.min().item():.3f}  max={ya.max().item():.3f}")

## Cell 6 — Datasets, DataLoaders, Loss Functions

In [ ]:
num_workers = 0

class Training(Dataset):
    def __init__(self, x_mix, yf_mix, ya_mix):
        assert x_mix.shape[0] == yf_mix.shape[0] == ya_mix.shape[0]
        self.x_mix  = x_mix
        self.yf_mix = yf_mix
        self.ya_mix = ya_mix

    def __len__(self): return self.x_mix.shape[0]
    def __getitem__(self, idx):
        return (self.x_mix[idx], self.yf_mix[idx], self.ya_mix[idx])


class Validation(Dataset):
    def __init__(self, x, yf, ya, c):
        assert x.shape[0] == yf.shape[0] == ya.shape[0] == c.shape[0]
        self.x  = x
        self.yf = yf
        self.ya = ya
        self.c  = c

    def __len__(self): return self.x.shape[0]
    def __getitem__(self, idx): return self.x[idx], self.yf[idx], self.ya[idx], self.c[idx]


def get_dataloaders(train_ds, val_ds, batch_size, num_workers, seed):
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        drop_last=True, num_workers=num_workers, generator=g)
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        drop_last=False, num_workers=num_workers)
    return train_loader, val_loader


class FatigueLoss(nn.Module):
    """Asymmetric Huber: penalises overprediction (unsafe) more."""
    def __init__(self, delta=0.5, overpredict_weight=1.5):
        super().__init__()
        self.delta = delta
        self.w_over = overpredict_weight

    def forward(self, pred, target):
        error = pred - target
        abs_error = error.abs()
        quadratic = 0.5 * error ** 2
        linear    = self.delta * (abs_error - 0.5 * self.delta)
        huber     = torch.where(abs_error <= self.delta, quadratic, linear)
        weight    = torch.where(error > 0, self.w_over, 1.0)
        return (weight * huber).mean()

## Cell 7 — Group-Aware Data Splitting (NEW)

In [ ]:
def split_data_grouped(x, yf, ya, c, seed, feature_cols,
                       train_frac=0.70, val_frac=0.15,
                       group_col_names=None):
    """
    Split by geometry groups so no parametric cluster straddles
    train / val / test.
    """
    n = len(x)
    test_frac = 1.0 - train_frac - val_frac
    x_np = x.numpy() if isinstance(x, torch.Tensor) else np.array(x)

    # Auto-detect discrete columns if not specified
    if group_col_names is None:
        group_cols_idx = []
        for col in range(x_np.shape[1]):
            n_unique = len(np.unique(np.round(x_np[:, col], 6)))
            if n_unique <= 30:
                group_cols_idx.append(col)
    else:
        group_cols_idx = [feature_cols.index(n) for n in group_col_names
                          if n in feature_cols]

    # Build group labels
    group_keys = np.round(x_np[:, group_cols_idx], 6)
    group_labels = np.array([hash(tuple(row)) for row in group_keys])

    unique_groups = np.unique(group_labels)
    group_map = {g: i for i, g in enumerate(unique_groups)}
    groups = np.array([group_map[g] for g in group_labels])

    n_groups = len(unique_groups)
    print(f'  Found {n_groups} geometry groups from {len(group_cols_idx)} columns')

    # Split 1: train vs (val+test) by group
    gss1 = GroupShuffleSplit(n_splits=1,
                            test_size=val_frac + test_frac,
                            random_state=seed)
    indices = np.arange(n)
    train_idx, valtest_idx = next(gss1.split(indices, groups=groups))

    # Split 2: val vs test by group
    relative_test = test_frac / (val_frac + test_frac)
    valtest_groups = groups[valtest_idx]
    gss2 = GroupShuffleSplit(n_splits=1,
                            test_size=relative_test,
                            random_state=seed)
    vt_local = np.arange(len(valtest_idx))
    val_local, test_local = next(gss2.split(vt_local, groups=valtest_groups))

    val_idx  = valtest_idx[val_local]
    test_idx = valtest_idx[test_local]

    # Verify no leakage
    train_groups = set(groups[train_idx])
    val_groups   = set(groups[val_idx])
    test_groups  = set(groups[test_idx])
    assert train_groups.isdisjoint(val_groups),  'Group leak: train ∩ val'
    assert train_groups.isdisjoint(test_groups), 'Group leak: train ∩ test'
    assert val_groups.isdisjoint(test_groups),   'Group leak: val ∩ test'

    print(f'  Groups  — Train: {len(train_groups)}  '
          f'Val: {len(val_groups)}  Test: {len(test_groups)}')
    print(f'  Samples — Train: {len(train_idx)}  '
          f'Val: {len(val_idx)}  Test: {len(test_idx)}')
    print(f'  ✓ No group leakage')

    return (torch.tensor(train_idx), torch.tensor(val_idx),
            torch.tensor(test_idx))


def mixup(x, yf, ya, alpha=alpha, seed=None):
    """Mixup with variable seed — different augmentation each epoch."""
    if seed is not None:
        np.random.seed(seed)
    N     = x.size(0)
    lam   = np.random.beta(alpha, alpha, size=N)
    lam   = np.maximum(lam, 1 - lam)
    lam_t = torch.tensor(lam, dtype=torch.float32).unsqueeze(1)
    idx_p = torch.randperm(N)

    x_mix  = lam_t * x  + (1 - lam_t) * x[idx_p]
    yf_mix = lam_t * yf + (1 - lam_t) * yf[idx_p]
    ya_mix = lam_t * ya + (1 - lam_t) * ya[idx_p]
    return x_mix, yf_mix, ya_mix


# Run the split
train_idx, val_idx, test_idx = split_data_grouped(
    x, yf, ya, c, seed=seed, feature_cols=feature_cols)

print(f'\n  Train : {len(train_idx):>3} samples  ({len(train_idx)/len(x)*100:.0f}%)')
print(f'  Val   : {len(val_idx):>3} samples  ({len(val_idx)/len(x)*100:.0f}%)')
print(f'  Test  : {len(test_idx):>3} samples  ({len(test_idx)/len(x)*100:.0f}%)')
for col_i, col in enumerate(target_col):
    tr_min, tr_max = y[train_idx, col_i].min(), y[train_idx, col_i].max()
    va_min, va_max = y[val_idx, col_i].min(), y[val_idx, col_i].max()
    te_min, te_max = y[test_idx, col_i].min(), y[test_idx, col_i].max()
    flag = '  ⚠ extrapolation' if (va_max > tr_max or te_max > tr_max) else ''
    print(f'  {col} range — '
          f'Train [{tr_min:.2f}, {tr_max:.2f}]  '
          f'Val [{va_min:.2f}, {va_max:.2f}]  '
          f'Test [{te_min:.2f}, {te_max:.2f}]{flag}')

## Cell 8 — Model Architecture (LayerNorm + skip + dropout)

In [ ]:
class Sohoite_Force(nn.Module):
    def __init__(self, input_dim, n_force=n_force, dropout=dropout):
        super().__init__()
        self.hidden1 = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.hidden2 = nn.Sequential(
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.skip  = nn.Linear(input_dim, 256)
        self.heads = nn.ModuleList([nn.Linear(256, 1) for _ in range(n_force)])
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        h1 = self.hidden1(x)
        h2 = self.hidden2(h1)
        h2 = h2 + self.skip(x)
        return torch.cat([head(h2) for head in self.heads], dim=1)


class Sohoite_Fatigue(nn.Module):
    def __init__(self, input_dim, n_force=n_force, n_fatigue=n_fatigue, dropout=dropout):
        super().__init__()
        combined_dim = input_dim + n_force
        self.hidden1 = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.hidden2 = nn.Sequential(
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.skip  = nn.Linear(combined_dim, 128)
        self.heads = nn.ModuleList([nn.Linear(128, 1) for _ in range(n_fatigue)])
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, Force_P):
        combined = torch.cat([x, Force_P], dim=1)
        h1 = self.hidden1(combined)
        h2 = self.hidden2(h1)
        h2 = h2 + self.skip(combined)
        return torch.cat([head(h2) for head in self.heads], dim=1)

## Cell 9 — Training & Evaluation Functions

In [ ]:
def train_force_epoch(force_model, loader, optimizer, device,
                      x_mean, x_std, yf_mean, yf_std):
    force_model.train()
    criterion  = nn.HuberLoss(delta=1.0)
    total_loss = 0.0
    for x_mix, yf_mix, _ in loader:
        x_mix  = x_mix.to(device)
        yf_mix = yf_mix.to(device)
        optimizer.zero_grad()
        pred_forces = force_model(x_mix)
        loss = criterion(pred_forces, yf_mix)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(force_model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(x_mix)
    return total_loss / len(loader.dataset)


def train_fatigue_epoch(force_model, fatigue_model, loader, optimizer, device,
                        x_mean, x_std, yf_mean, yf_std, ya_mean, ya_std):
    force_model.eval()
    fatigue_model.train()
    criterion  = FatigueLoss(delta=0.5, overpredict_weight=1.5)
    total_loss = 0.0
    for x_mix, _, ya_mix in loader:
        x_mix  = x_mix.to(device)
        ya_mix = ya_mix.to(device)
        optimizer.zero_grad()
        with torch.no_grad():
            pred_forces = force_model(x_mix)
        pred_fatigue = fatigue_model(x_mix, pred_forces)
        loss = criterion(pred_fatigue, ya_mix)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fatigue_model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(x_mix)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_force(force_model, loader, device):
    force_model.eval()
    criterion  = nn.MSELoss()
    total_loss = 0.0
    all_pf, all_tf = [], []
    for x_batch, yf_batch, ya_batch, _ in loader:
        x_batch  = x_batch.to(device)
        yf_batch = yf_batch.to(device)
        pred_f = force_model(x_batch)
        total_loss += criterion(pred_f, yf_batch).item() * len(x_batch)
        all_pf.append(pred_f.cpu())
        all_tf.append(yf_batch.cpu())
    return (total_loss / len(loader.dataset),
            torch.cat(all_pf), torch.cat(all_tf))


@torch.no_grad()
def evaluate_fatigue(force_model, fatigue_model, loader, device):
    force_model.eval()
    fatigue_model.eval()
    criterion  = nn.MSELoss()
    total_loss = 0.0
    all_pa, all_ta = [], []
    for x_batch, yf_batch, ya_batch, _ in loader:
        x_batch  = x_batch.to(device)
        ya_batch = ya_batch.to(device)
        pred_f  = force_model(x_batch)
        pred_fa = fatigue_model(x_batch, pred_f)
        total_loss += criterion(pred_fa, ya_batch).item() * len(x_batch)
        all_pa.append(pred_fa.cpu())
        all_ta.append(ya_batch.cpu())
    return (total_loss / len(loader.dataset),
            torch.cat(all_pa), torch.cat(all_ta))


def tester(y_pred_norm, y_true_norm, y_mean, y_std, cols):
    y_pred = (y_pred_norm * y_std + y_mean).numpy()
    y_true = (y_true_norm * y_std + y_mean).numpy()
    results = {}
    for i, col in enumerate(cols):
        err    = y_pred[:, i] - y_true[:, i]
        mae    = float(np.mean(np.abs(err)))
        rmse   = float(np.sqrt(np.mean(err ** 2)))
        ss_res = np.sum(err ** 2)
        ss_tot = np.sum((y_true[:, i] - y_true[:, i].mean()) ** 2)
        r2     = float(1 - ss_res / (ss_tot + 1e-8))
        results[col] = {'mae': mae, 'rmse': rmse, 'r2': r2}
    mean_mae = float(np.mean([v['mae'] for v in results.values()]))
    return results, mean_mae

## Cell 10 — Training Loop (both stages)

In [ ]:
device = torch.device(
    'mps'  if torch.backends.mps.is_available()  else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)

# Normalize using training set only
x_tr  = x[train_idx];  yf_tr  = yf[train_idx];  ya_tr = ya[train_idx]
x_val = x[val_idx];    yf_val = yf[val_idx];     ya_val = ya[val_idx]
x_te  = x[test_idx];   yf_te  = yf[test_idx];    ya_te = ya[test_idx]
c_val = c[val_idx];    c_te   = c[test_idx]

x_mean  = x_tr.mean(dim=0);   x_std  = x_tr.std(dim=0).clamp(min=1e-8)
yf_mean = yf_tr.mean(dim=0);  yf_std = yf_tr.std(dim=0).clamp(min=1e-8)
ya_mean = ya_tr.mean(dim=0);  ya_std = ya_tr.std(dim=0).clamp(min=1e-8)

x_tr_norm  = (x_tr  - x_mean) / x_std
x_val_norm = (x_val - x_mean) / x_std
x_te_norm  = (x_te  - x_mean) / x_std
yf_tr_norm  = (yf_tr  - yf_mean) / yf_std
yf_val_norm = (yf_val - yf_mean) / yf_std
yf_te_norm  = (yf_te  - yf_mean) / yf_std
ya_tr_norm  = (ya_tr  - ya_mean) / ya_std
ya_val_norm = (ya_val - ya_mean) / ya_std
ya_te_norm  = (ya_te  - ya_mean) / ya_std

val_ds  = Validation(x_val_norm, yf_val_norm, ya_val_norm, c_val)
test_ds = Validation(x_te_norm,  yf_te_norm,  ya_te_norm,  c_te)
val_loader  = DataLoader(val_ds,  batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

# Models
force_model   = Sohoite_Force(input_dim=x.shape[1]).to(device)
fatigue_model = Sohoite_Fatigue(input_dim=x.shape[1]).to(device)

# ── Stage 1: ForceNet ────────────────────────────────────────────
optimizer_force = optim.AdamW(force_model.parameters(), lr=1e-4, weight_decay=5e-3)

best_force_val    = float('inf')
best_force_state  = None
epochs_no_improve = 0
stopped_force     = epochs
force_train_hist  = []
force_val_hist    = []

print("  Stage 1 — ForceNet")
print(f'  {"Epoch":>6} | {"Train Loss":>12} | {"Val Loss":>10} |')
print(f'  {"-"*40}')

for epoch in range(1, epochs + 1):
    x_mix, yf_mix, ya_mix = mixup(x_tr_norm, yf_tr_norm, ya_tr_norm,
                                   alpha=alpha, seed=epoch)
    train_ds        = Training(x_mix, yf_mix, ya_mix)
    train_loader, _ = get_dataloaders(train_ds, val_ds,
                                      batch_size=batch_size, num_workers=0, seed=epoch)

    tr_loss        = train_force_epoch(force_model, train_loader, optimizer_force,
                                       device, x_mean, x_std, yf_mean, yf_std)
    val_loss, _, _ = evaluate_force(force_model, val_loader, device)

    force_train_hist.append(tr_loss)
    force_val_hist.append(val_loss)

    if val_loss < best_force_val:
        best_force_val   = val_loss
        best_force_state = {k: v.clone() for k, v in force_model.state_dict().items()}
        epochs_no_improve = 0
        mark = '*'
    else:
        epochs_no_improve += 1
        mark = ' '

    if epoch % 50 == 0 or epoch == 1:
        print(f'  {epoch:>6} | {tr_loss:>12.6f} | {val_loss:>10.6f} | {mark}')

    if epochs_no_improve >= patience:
        stopped_force = epoch
        break

print(f'\n  Stopped : epoch {stopped_force}  |  Best val loss : {best_force_val:.6f}')
force_model.load_state_dict(best_force_state)

for p in force_model.parameters():
    p.requires_grad = False

# ── Stage 2: FatigueNet ─────────────────────────────────────────
optimizer_fatigue = optim.AdamW(fatigue_model.parameters(), lr=1e-4, weight_decay=5e-3)

best_fatigue_val    = float('inf')
best_fatigue_state  = None
epochs_no_improve   = 0
stopped_fatigue     = epochs
fatigue_train_hist  = []
fatigue_val_hist    = []

print("\n  Stage 2 — FatigueNet (ForceNet frozen)")
print(f'  {"Epoch":>6} | {"Train Loss":>12} | {"Val Loss":>10} |')
print(f'  {"-"*40}')

for epoch in range(1, epochs + 1):
    x_mix, yf_mix, ya_mix = mixup(x_tr_norm, yf_tr_norm, ya_tr_norm,
                                   alpha=alpha, seed=epoch)
    train_ds        = Training(x_mix, yf_mix, ya_mix)
    train_loader, _ = get_dataloaders(train_ds, val_ds,
                                      batch_size=batch_size, num_workers=0, seed=epoch)

    tr_loss        = train_fatigue_epoch(force_model, fatigue_model, train_loader,
                                         optimizer_fatigue, device,
                                         x_mean, x_std, yf_mean, yf_std, ya_mean, ya_std)
    val_loss, _, _ = evaluate_fatigue(force_model, fatigue_model, val_loader, device)

    fatigue_train_hist.append(tr_loss)
    fatigue_val_hist.append(val_loss)

    if val_loss < best_fatigue_val:
        best_fatigue_val   = val_loss
        best_fatigue_state = {k: v.clone() for k, v in fatigue_model.state_dict().items()}
        epochs_no_improve  = 0
        mark = '*'
    else:
        epochs_no_improve += 1
        mark = ' '

    if epoch % 50 == 0 or epoch == 1:
        print(f'  {epoch:>6} | {tr_loss:>12.6f} | {val_loss:>10.6f} | {mark}')

    if epochs_no_improve >= patience:
        stopped_fatigue = epoch
        break

print(f'\n  Stopped : epoch {stopped_fatigue}  |  Best val loss : {best_fatigue_val:.6f}')
fatigue_model.load_state_dict(best_fatigue_state)

## Cell 11 — Evaluation & Residuals

In [ ]:
val_floss,  pf_val, tf_val = evaluate_force(force_model,   val_loader,  device)
test_floss, pf_te,  tf_te  = evaluate_force(force_model,   test_loader, device)
val_aloss,  pa_val, ta_val = evaluate_fatigue(force_model, fatigue_model, val_loader,  device)
test_aloss, pa_te,  ta_te  = evaluate_fatigue(force_model, fatigue_model, test_loader, device)

val_force_res,  val_force_mae  = tester(pf_val, tf_val, yf_mean, yf_std, force_cols)
test_force_res, test_force_mae = tester(pf_te,  tf_te,  yf_mean, yf_std, force_cols)
val_fat_res,    val_fat_mae    = tester(pa_val, ta_val, ya_mean, ya_std, fatigue_cols)
test_fat_res,   test_fat_mae   = tester(pa_te,  ta_te,  ya_mean, ya_std, fatigue_cols)

pf_val_w = (pf_val * yf_std + yf_mean).numpy(); tf_val_w = (tf_val * yf_std + yf_mean).numpy()
pf_te_w  = (pf_te  * yf_std + yf_mean).numpy(); tf_te_w  = (tf_te  * yf_std + yf_mean).numpy()
pa_val_w = (pa_val * ya_std + ya_mean).numpy(); ta_val_w = (ta_val * ya_std + ya_mean).numpy()
pa_te_w  = (pa_te  * ya_std + ya_mean).numpy(); ta_te_w  = (ta_te  * ya_std + ya_mean).numpy()

# Print results
print(f'\n  {"Output":>12} | {"Val MAE":>8} | {"Val RMSE":>9} | {"Val R²":>7} | '
      f'{"Test MAE":>9} | {"Test RMSE":>10} | {"Test R²":>8}')
print(f'  {"-"*68}')
print(f'  --- Forces ---')
for col in force_cols:
    print(f'  {col:>12} | '
          f'{val_force_res[col]["mae"]:>8.4f} | '
          f'{val_force_res[col]["rmse"]:>9.4f} | '
          f'{val_force_res[col]["r2"]:>7.4f} | '
          f'{test_force_res[col]["mae"]:>9.4f} | '
          f'{test_force_res[col]["rmse"]:>10.4f} | '
          f'{test_force_res[col]["r2"]:>8.4f}')
print(f'  --- Fatigue ---')
for col in fatigue_cols:
    print(f'  {col:>12} | '
          f'{val_fat_res[col]["mae"]:>8.4f} | '
          f'{val_fat_res[col]["rmse"]:>9.4f} | '
          f'{val_fat_res[col]["r2"]:>7.4f} | '
          f'{test_fat_res[col]["mae"]:>9.4f} | '
          f'{test_fat_res[col]["rmse"]:>10.4f} | '
          f'{test_fat_res[col]["r2"]:>8.4f}')

# Residuals
print('\n  Residuals')
print('  ' + '-'*65)
print(f'  {"Output":>10} | {"Set":>5} | {"Bias":>8} | {"Std":>8} | {"Max Err":>8}')
print('  ' + '-'*65)
for group_name, cols, pv, tv, pt, tt in [
    ('Forces',  force_cols,   pf_val_w, tf_val_w, pf_te_w, tf_te_w),
    ('Fatigue', fatigue_cols, pa_val_w, ta_val_w, pa_te_w, ta_te_w),
]:
    print(f'  --- {group_name} ---')
    for i, col in enumerate(cols):
        for label, yp, yt in [('Val', pv, tv), ('Test', pt, tt)]:
            err = yp[:, i] - yt[:, i]
            print(f'  {col:>10} | {label:>5} | {err.mean():>+8.4f} | '
                  f'{err.std():>8.4f} | {np.abs(err).max():>8.4f}')
print('  ' + '-'*65)

## Cell 12 — Plots (loss curves, scatter, fatigue)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Training & Validation Loss', fontweight='bold')
for ax, train_h, val_h, stopped, title in [
    (axes[0], force_train_hist,   force_val_hist,   stopped_force,   'Stage 1 — Forces'),
    (axes[1], fatigue_train_hist, fatigue_val_hist, stopped_fatigue, 'Stage 2 — Fatigue'),
]:
    ax.plot(train_h, label='Train', color='steelblue', alpha=0.8)
    ax.plot(val_h,   label='Val',   color='orange',    alpha=0.8)
    ax.axvline(stopped - patience, color='red', linestyle='--', label=f'Early stop (ep {stopped})')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (normalised)')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

n_f = len(force_cols)
fig, axes = plt.subplots(2, (n_f + 1) // 2, figsize=(5 * ((n_f + 1) // 2), 9))
fig.suptitle('Force predictions — val vs test', fontweight='bold')
axes = axes.flatten()
for i, col in enumerate(force_cols):
    ax = axes[i]
    ax.scatter(tf_val_w[:, i], pf_val_w[:, i], alpha=0.7, color='steelblue', s=40, label='Val')
    ax.scatter(tf_te_w[:,  i], pf_te_w[:,  i], alpha=0.9, color='orange',    s=50, label='Test', marker='^')
    lo = min(tf_val_w[:, i].min(), tf_te_w[:, i].min()) - 1
    hi = max(tf_val_w[:, i].max(), tf_te_w[:, i].max()) + 1
    ax.plot([lo, hi], [lo, hi], 'r--')
    ax.set_title(col); ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
for ax in axes[n_f:]: ax.set_visible(False)
plt.tight_layout()
plt.savefig('force_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

n_a = len(fatigue_cols)
fig, axes = plt.subplots(1, n_a, figsize=(5 * n_a, 4))
fig.suptitle('Fatigue life predictions — val vs test', fontweight='bold')
for i, col in enumerate(fatigue_cols):
    ax = axes[i]
    ax.scatter(ta_val_w[:, i], pa_val_w[:, i], alpha=0.7, color='steelblue', s=40, label='Val')
    ax.scatter(ta_te_w[:,  i], pa_te_w[:,  i], alpha=0.9, color='orange',    s=50, label='Test', marker='^')
    lo = min(ta_val_w[:, i].min(), ta_te_w[:, i].min()) - 1
    hi = max(ta_val_w[:, i].max(), ta_te_w[:, i].max()) + 1
    ax.plot([lo, hi], [lo, hi], 'r--')
    ax.set_title(col); ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fatigue_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 13 — Experiment Logger

In [ ]:
log_file   = 'experiment_log.csv'
run_number = 0
if os.path.isfile(log_file):
    df_log     = pd.read_csv(log_file)
    run_number = len(df_log)

dropout_val = 0.0
for module in force_model.modules():
    if isinstance(module, nn.Dropout):
        dropout_val = module.p; break

norm_type = 'None'
for module in force_model.modules():
    if isinstance(module, nn.LayerNorm):
        norm_type = 'LayerNorm'; break
    elif isinstance(module, nn.BatchNorm1d):
        norm_type = 'BatchNorm1d'; break

has_skip = hasattr(force_model, 'skip')

for m in force_model.hidden1:
    if isinstance(m, nn.Linear): w1 = m.out_features; break
for m in force_model.hidden2:
    if isinstance(m, nn.Linear): w2 = m.out_features; break
force_widths = f'{w1} → {w2}'

for m in fatigue_model.hidden1:
    if isinstance(m, nn.Linear): fw1 = m.out_features; break
for m in fatigue_model.hidden2:
    if isinstance(m, nn.Linear): fw2 = m.out_features; break
fatigue_widths = f'{fw1} → {fw2}'
if has_skip:
    force_widths   += ' + skip'
    fatigue_widths += ' + skip'

n_params_force   = sum(p.numel() for p in force_model.parameters())
n_params_fatigue = sum(p.numel() for p in fatigue_model.parameters())

run_config = {
    'run_number'       : run_number + 1,
    'timestamp'        : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'split_method'     : 'grouped',
    'loss_force'       : 'HuberLoss(delta=1.0)',
    'loss_fatigue'     : 'FatigueLoss(delta=0.5,overpredict=1.5)',
    'hidden_force'     : force_widths,
    'hidden_fatigue'   : fatigue_widths,
    'norm_type'        : norm_type,
    'skip_connection'  : has_skip,
    'n_params_force'   : n_params_force,
    'n_params_fatigue' : n_params_fatigue,
    'optimizer'        : type(optimizer_force).__name__,
    'lr'               : optimizer_force.param_groups[0]['lr'],
    'weight_decay'     : optimizer_force.param_groups[0].get('weight_decay', 0),
    'dropout'          : dropout_val,
    'lambda_pinn'      : Lambda,
    'CHECK'            : str(CHECK),
    'epochs_max'       : epochs,
    'stopped_force'    : stopped_force,
    'stopped_fatigue'  : stopped_fatigue,
    'best_force_val'   : round(best_force_val, 6),
    'best_fatigue_val' : round(best_fatigue_val, 6),
    'patience'         : patience,
    'batch_size'       : batch_size,
    'alpha_mixup'      : alpha,
    'n_force_outputs'  : n_force,
    'n_fatigue_outputs': n_fatigue,
    'dimensionless'    : True,
    'n_dim_features'   : len(dimensionless_cols),
    'seed'             : seed,
    'n_total'          : len(x),
    'n_train'          : len(train_idx),
    'n_val'            : len(val_idx),
    'n_test'           : len(test_idx),
    'pct_train'        : round(len(train_idx)/len(x)*100, 1),
    'pct_val'          : round(len(val_idx)/len(x)*100,   1),
    'pct_test'         : round(len(test_idx)/len(x)*100,  1),
    'notes'            : '',
}

run_metrics = {}
for col in force_cols:
    for split, res in [('val', val_force_res), ('test', test_force_res)]:
        run_metrics[f'{col}_{split}_mae']  = round(res[col]['mae'],  4)
        run_metrics[f'{col}_{split}_rmse'] = round(res[col]['rmse'], 4)
        run_metrics[f'{col}_{split}_mse']  = round(res[col]['rmse']**2, 4)
        run_metrics[f'{col}_{split}_r2']   = round(res[col]['r2'],   4)

for col in fatigue_cols:
    for split, res in [('val', val_fat_res), ('test', test_fat_res)]:
        run_metrics[f'{col}_{split}_mae']  = round(res[col]['mae'],  4)
        run_metrics[f'{col}_{split}_rmse'] = round(res[col]['rmse'], 4)
        run_metrics[f'{col}_{split}_mse']  = round(res[col]['rmse']**2, 4)
        run_metrics[f'{col}_{split}_r2']   = round(res[col]['r2'],   4)

for split in ['val', 'test']:
    force_r2s  = [run_metrics[f'{c}_{split}_r2']  for c in force_cols]
    force_maes = [run_metrics[f'{c}_{split}_mae'] for c in force_cols]
    fat_r2s    = [run_metrics[f'{c}_{split}_r2']  for c in fatigue_cols]
    fat_maes   = [run_metrics[f'{c}_{split}_mae'] for c in fatigue_cols]
    run_metrics[f'force_mean_{split}_r2']    = round(np.mean(force_r2s),  4)
    run_metrics[f'force_mean_{split}_mae']   = round(np.mean(force_maes), 4)
    run_metrics[f'fatigue_mean_{split}_r2']  = round(np.mean(fat_r2s),    4)
    run_metrics[f'fatigue_mean_{split}_mae'] = round(np.mean(fat_maes),   4)

row = {**run_config, **run_metrics}
if os.path.isfile(log_file):
    existing = pd.read_csv(log_file)
    new_row  = pd.DataFrame([row])
    combined = pd.concat([existing, new_row], ignore_index=True)
else:
    combined = pd.DataFrame([row])
combined.to_csv(log_file, index=False)

print('=' * 70)
print(f'  Run {run_config["run_number"]} logged → {log_file}')
print('=' * 70)
print(f'  Split         : {run_config["split_method"]}  '
      f'({run_config["pct_train"]}/{run_config["pct_val"]}/{run_config["pct_test"]}%)')
print(f'  Loss          : force={run_config["loss_force"]}  '
      f'fatigue={run_config["loss_fatigue"]}')
print(f'  Force arch    : {force_widths}  ({n_params_force:,} params)')
print(f'  Fatigue arch  : {fatigue_widths}  ({n_params_fatigue:,} params)')
print(f'  Norm          : {norm_type}   Skip: {has_skip}')
print(f'  Optimizer     : {run_config["optimizer"]}  lr={run_config["lr"]}  '
      f'wd={run_config["weight_decay"]}')
print(f'  Dropout       : {dropout_val}   Mixup α: {alpha}')
print(f'  Dimensionless : {len(dimensionless_cols)} features added')
print(f'  Stopped       : force={stopped_force}  fatigue={stopped_fatigue}  / {epochs}')
print(f'  Best val loss : force={best_force_val:.6f}  fatigue={best_fatigue_val:.6f}')
print('-' * 70)
print(f'  {"Output":>12} | {"Val MAE":>8} | {"Val RMSE":>9} | {"Val R²":>7} | '
      f'{"Test MAE":>9} | {"Test RMSE":>10} | {"Test R²":>8}')
print(f'  {"-"*68}')
print(f'  --- Forces ---')
for col in force_cols:
    print(f'  {col:>12} | '
          f'{run_metrics[f"{col}_val_mae"]:>8.4f} | '
          f'{run_metrics[f"{col}_val_rmse"]:>9.4f} | '
          f'{run_metrics[f"{col}_val_r2"]:>7.4f} | '
          f'{run_metrics[f"{col}_test_mae"]:>9.4f} | '
          f'{run_metrics[f"{col}_test_rmse"]:>10.4f} | '
          f'{run_metrics[f"{col}_test_r2"]:>8.4f}')
print(f'  {"MEAN":>12} | '
      f'{run_metrics["force_mean_val_mae"]:>8.4f} | {"":>9} | '
      f'{run_metrics["force_mean_val_r2"]:>7.4f} | '
      f'{run_metrics["force_mean_test_mae"]:>9.4f} | {"":>10} | '
      f'{run_metrics["force_mean_test_r2"]:>8.4f}')
print(f'  --- Fatigue ---')
for col in fatigue_cols:
    print(f'  {col:>12} | '
          f'{run_metrics[f"{col}_val_mae"]:>8.4f} | '
          f'{run_metrics[f"{col}_val_rmse"]:>9.4f} | '
          f'{run_metrics[f"{col}_val_r2"]:>7.4f} | '
          f'{run_metrics[f"{col}_test_mae"]:>9.4f} | '
          f'{run_metrics[f"{col}_test_rmse"]:>10.4f} | '
          f'{run_metrics[f"{col}_test_r2"]:>8.4f}')
print(f'  {"MEAN":>12} | '
      f'{run_metrics["fatigue_mean_val_mae"]:>8.4f} | {"":>9} | '
      f'{run_metrics["fatigue_mean_val_r2"]:>7.4f} | '
      f'{run_metrics["fatigue_mean_test_mae"]:>9.4f} | {"":>10} | '
      f'{run_metrics["fatigue_mean_test_r2"]:>8.4f}')
print('=' * 70)

## Cell 14 — SHAP Analysis

In [ ]:
class ForceWrapper:
    def __init__(self, model, device):
        self.model  = model
        self.device = device
        self.model.eval()
    def __call__(self, X):
        with torch.no_grad():
            t = torch.tensor(X, dtype=torch.float32).to(self.device)
            return self.model(t).cpu().numpy()


class FatigueWrapper:
    def __init__(self, force_model, fatigue_model, device):
        self.force_model   = force_model
        self.fatigue_model = fatigue_model
        self.device        = device
        self.force_model.eval()
        self.fatigue_model.eval()
    def __call__(self, X):
        with torch.no_grad():
            t = torch.tensor(X, dtype=torch.float32).to(self.device)
            pred_f = self.force_model(t)
            pred_a = self.fatigue_model(t, pred_f)
            return pred_a.cpu().numpy()


n_background = min(200, len(x_tr_norm))
bg_idx       = np.random.choice(len(x_tr_norm), n_background, replace=False)
background   = x_tr_norm[bg_idx].numpy()

n_explain    = min(100, len(x_te_norm))
explain_idx  = np.random.choice(len(x_te_norm), n_explain, replace=False)
X_explain    = x_te_norm[explain_idx].numpy()

# ── Force SHAP ───────────────────────────────────────────────────
print('  Computing SHAP for ForceNet...')
force_fn        = ForceWrapper(force_model, device)
force_explainer = shap.KernelExplainer(force_fn, background)
force_shap_values = force_explainer.shap_values(X_explain)

all_force_shap = np.stack(force_shap_values, axis=0)
mean_abs_shap  = np.mean(np.abs(all_force_shap), axis=(0, 1))
sorted_idx     = np.argsort(mean_abs_shap)[::-1]
top_k          = 15

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(top_k), mean_abs_shap[sorted_idx[:top_k]][::-1],
        color='steelblue', alpha=0.8)
ax.set_yticks(range(top_k))
ax.set_yticklabels([feature_cols[i] for i in sorted_idx[:top_k]][::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Stage 1 — ForceNet: Feature Importance (all outputs)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('shap_force_summary.png', dpi=150, bbox_inches='tight')
plt.show()

for i, col in enumerate(force_cols):
    shap.summary_plot(force_shap_values[i], X_explain,
                      feature_names=feature_cols, plot_type='dot',
                      max_display=15, show=False)
    plt.title(f'Stage 1 — {col}')
    plt.tight_layout()
    plt.savefig(f'shap_force_{col.replace("|","").replace(" ","_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

# ── Fatigue SHAP ─────────────────────────────────────────────────
print('  Computing SHAP for FatigueNet (cascade)...')
fatigue_fn        = FatigueWrapper(force_model, fatigue_model, device)
fatigue_explainer = shap.KernelExplainer(fatigue_fn, background)
fatigue_shap_values = fatigue_explainer.shap_values(X_explain)

all_fat_shap  = np.stack(fatigue_shap_values, axis=0)
mean_abs_fat  = np.mean(np.abs(all_fat_shap), axis=(0, 1))
sorted_idx_f  = np.argsort(mean_abs_fat)[::-1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(top_k), mean_abs_fat[sorted_idx_f[:top_k]][::-1],
        color='darkorange', alpha=0.8)
ax.set_yticks(range(top_k))
ax.set_yticklabels([feature_cols[i] for i in sorted_idx_f[:top_k]][::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Stage 2 — Fatigue (cascade): Feature Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('shap_fatigue_summary.png', dpi=150, bbox_inches='tight')
plt.show()

for i, col in enumerate(fatigue_cols):
    shap.summary_plot(fatigue_shap_values[i], X_explain,
                      feature_names=feature_cols, plot_type='dot',
                      max_display=15, show=False)
    plt.title(f'Stage 2 — {col}')
    plt.tight_layout()
    plt.savefig(f'shap_fatigue_{col}.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── Top features table ───────────────────────────────────────────
print('\n' + '=' * 60)
print('  Top 10 Features by Mean |SHAP|')
print('=' * 60)
print(f'  {"Rank":>4} | {"Force Model":>25} | {"Fatigue (cascade)":>25}')
print(f'  {"-"*58}')
for rank in range(10):
    f_name = feature_cols[sorted_idx[rank]]
    f_val  = mean_abs_shap[sorted_idx[rank]]
    a_name = feature_cols[sorted_idx_f[rank]]
    a_val  = mean_abs_fat[sorted_idx_f[rank]]
    print(f'  {rank+1:>4} | {f_name:>18} {f_val:.4f} | '
          f'{a_name:>18} {a_val:.4f}')
print('=' * 60)

# ── Flag: dimensionless vs raw ───────────────────────────────────
print('\n  Dimensionless feature ranking:')
for rank in range(top_k):
    fname = feature_cols[sorted_idx[rank]]
    is_dim = '✓ dimensionless' if fname in dimensionless_cols else '  raw'
    print(f'    {rank+1:>2}. {fname:<25} {is_dim}')